# Actividad integradora — De los notebooks al pipeline automatizado

**Grupo 03 · Ingeniería de Datos · LCD · UCA Rosario · 2026**

**Fuente de datos:** API de VATSIM (`https://data.vatsim.net/v3/vatsim-data.json`)

Este notebook reúne la evidencia de cada paso de la actividad. Las celdas se
ejecutan en orden y no dependen de ningún estado previo.

La lógica operativa **no vive acá**: está en `scripts/` y en el proyecto `dbt/`.
Este notebook solo verifica y documenta.

## Paso 0 — Estructura del workspace

Una única raíz por componente. `dbt/` y `scripts/` son enlaces a `dags/`, que es
la carpeta que el servidor monta dentro del contenedor de Airflow: así existe
una sola copia real de cada archivo.

In [1]:
import subprocess


def sh(cmd: str) -> None:
    """Ejecuta un comando en el workspace y muestra su salida."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                       cwd="/home/jovyan/work")
    print(r.stdout or "", r.stderr or "")


print("=== Raiz del proyecto ===")
sh("ls -la | grep -v '^total'")

print("=== Arbol de dags/ (lo que ve Airflow) ===")
sh("find dags -not -path '*__pycache__*' -not -path '*/target/*' "
   "-not -path '*/logs/*' -not -path '*.ipynb_checkpoints*' | sort")

=== Raiz del proyecto ===
drwsrwsr-x 10 jovyan users    18 Oct 23 21:10 .
drwsrws---  1 jovyan users     7 Oct 23 20:54 ..
-rw-r--r--  1 jovyan users   122 Oct 23 20:37 .env.example
drwxr-sr-x  2 jovyan users    17 Oct 23 20:25 .ipynb_checkpoints
-rw-r--r--  1 jovyan users    41 Oct 12 16:52 .user.yml
drwxr-sr-x 10 jovyan users    15 Oct 23 00:09 INGDD-UCA2026
drwxr-sr-x  2 jovyan users     3 Oct 23 21:00 __pycache__
drwxr-sr-x  4 jovyan users     7 Oct 23 04:09 _backup_dbt_pre_rename
-rw-r--r--  1 jovyan users 31393 Oct 23 21:01 clase3practica.ipynb
drwxrwxr-x  6 jovyan root     12 Oct 23 20:58 dags
lrwxrwxrwx  1 jovyan users     8 Oct 23 21:00 dbt -> dags/dbt
-rw-r--r--  1 jovyan users  3482 Aug 27 17:35 entorno.py
drwxr-sr-x  3 jovyan users     5 Oct 16 22:27 logs
drwxr-sr-x  3 jovyan users    11 Oct 23 21:10 notebooks
-rw-r--r--  1 jovyan users 93938 Oct 23 21:10 prueba.ipynb
-rw-r--r--  1 jovyan users  1610 Oct 23 01:43 respaldo_pipeline_grupo03_20261023_203332.py.txt
drwxr-sr-x  

## Paso 2 — Prueba aislada de `scripts/ingesta.py`

Se ejecuta fuera de Jupyter, como módulo. Se corre **dos veces seguidas** para
verificar la idempotencia: la segunda no debe escribir filas.

In [2]:
def probar(titulo: str, cmd: str) -> None:
    """Ejecuta un comando y muestra salida y codigo de retorno."""
    print(f"\n{'=' * 58}\n{titulo}\n{'=' * 58}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                       cwd="/home/jovyan/work")
    print(r.stdout or "")
    if r.stderr:
        print("STDERR:", r.stderr[-800:])
    print(f"--- exit code: {r.returncode} ---")


probar("Primera corrida", "python -m scripts.ingesta")
probar("Segunda corrida (idempotencia)", "python -m scripts.ingesta")


Primera corrida
Configuracion validada correctamente.
Filas extraidas: 1493 (snapshot 2026-10-23T21:10:54.0264802Z)
Ingeridas 1493 filas (watermark previo=2026-10-23T20:49:09.0245399Z).
Filas ingeridas: 1493

--- exit code: 0 ---

Segunda corrida (idempotencia)
Configuracion validada correctamente.
Sin snapshot nuevo (watermark=2026-10-23T21:10:54.0264802Z).
Sin novedades (watermark=2026-10-23T21:10:54.0264802Z).
Filas ingeridas: 0

--- exit code: 0 ---


**Resultado esperado:** ambas con código de salida 0. La segunda informa
`Sin snapshot nuevo` y escribe 0 filas, porque el watermark sobre `snapshot_ts`
detecta que la foto ya fue ingerida.

Controles que cumple el script:

- No contiene credenciales escritas en el código.
- Falla con mensaje claro si falta una variable obligatoria.
- Valida el código de estado de la respuesta de la API.
- Controla respuestas vacías.
- Informa cuántos registros obtuvo y cuántos escribió.
- Dos ejecuciones seguidas no duplican datos.

## Paso 4 — Proyecto dbt: debug, run y test

Se configuran las variables que el perfil resuelve con `env_var()` y se ejecuta
la cadena completa.

In [3]:
import os
import sys
from pathlib import Path
from urllib.parse import urlsplit

sys.path.insert(0, "/home/jovyan/work")
from entorno import STORAGE_OPTIONS, DBT_DIR, BUCKET

os.environ["AWS_ACCESS_KEY_ID"] = STORAGE_OPTIONS["AWS_ACCESS_KEY_ID"]
os.environ["AWS_SECRET_ACCESS_KEY"] = STORAGE_OPTIONS["AWS_SECRET_ACCESS_KEY"]
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["MINIO_BUCKET"] = BUCKET

url = urlsplit(STORAGE_OPTIONS["AWS_ENDPOINT_URL"])
os.environ["DBT_S3_ENDPOINT"] = url.netloc
os.environ["DBT_S3_USE_SSL"] = str(url.scheme == "https").lower()
os.environ["DBT_DUCKDB_PATH"] = str(Path(DBT_DIR) / "warehouse.duckdb")

print("Proyecto dbt:", DBT_DIR)
print("Bucket      :", BUCKET)

Proyecto dbt: /home/jovyan/work/dbt
Bucket      : grupo03


In [4]:
for comando in ["dbt debug", "dbt run", "dbt test"]:
    print(f"\n{'=' * 58}\n{comando}\n{'=' * 58}")
    r = subprocess.run(comando.split() + ["--profiles-dir", "."],
                       cwd=DBT_DIR, capture_output=True, text=True)
    print(r.stdout[-2500:])
    print(f"--- exit code: {r.returncode} ---")


dbt debug
21:11:29  Running with dbt=1.12.0
21:11:29  dbt version: 1.12.0
21:11:29  python version: 3.13.14
21:11:29  python path: /opt/conda/bin/python3.13
21:11:29  os info: Linux-7.0.14-8-pve-x86_64-with-glibc2.39
21:11:29  Using profiles dir at .
21:11:29  Using profiles.yml file at ./profiles.yml
21:11:29  Using dbt_project.yml file at /home/jovyan/work/dags/dbt/dbt_project.yml
21:11:29  adapter type: duckdb
21:11:29  adapter version: 1.10.1
21:11:29  Configuration:
21:11:29    profiles.yml file [OK found and valid]
21:11:29    dbt_project.yml file [OK found and valid]
21:11:29  Required dependencies:
21:11:29   - git [OK found]

21:11:29  Connection:
21:11:29    database: warehouse
21:11:29    schema: main
21:11:29    path: /home/jovyan/work/dbt/warehouse.duckdb
21:11:29    config_options: None
21:11:29    extensions: ['httpfs', 'delta']
21:11:29    settings: {}
21:11:29    external_root: .
21:11:29    use_credential_provider: None
21:11:29    attach: None
21:11:29    filesystem

**Modelos y capas**

| Modelo | Capa | Materialización |
|---|---|---|
| `brz_vatsim` | staging | vista |
| `stg_vatsim` | staging | vista |
| `fct_trafico_snapshot` | marts | tabla |

`staging` hace tipado y limpieza 1:1 con la fuente, sin lógica de negocio.
`marts` contiene el modelo orientado al consumo, materializado como tabla para
que Metabase lo lea rápido y sin depender del acceso al lake.

## Paso 3 — Prueba aislada de `scripts/publicar_gold.py`

Lee la capa gold desde DuckDB y la publica en PostgreSQL, donde la consume
Metabase.

In [ ]:
probar("Publicacion de la capa gold", "python -m scripts.publicar_gold")

In [5]:
# Verificacion en el destino: la tabla publicada en PostgreSQL.
from sqlalchemy import create_engine, text

from entorno import pg_url

motor = create_engine(pg_url())
with motor.connect() as con:
    filas = con.execute(
        text("select count(*) from grupo03.fct_trafico_snapshot")
    ).scalar_one()
    muestra = con.execute(text(
        "select snapshot_ts, pilotos_conectados, var_pct_pilotos "
        "from grupo03.fct_trafico_snapshot order by snapshot_ts desc limit 5"
    )).fetchall()
motor.dispose()

print("Filas en warehouse.grupo03.fct_trafico_snapshot:", filas)
for fila in muestra:
    print(" ", fila)

Filas en warehouse.grupo03.fct_trafico_snapshot: 11
  (datetime.datetime(2026, 10, 23, 3, 51, 54, 25855), 644, -1.53)
  (datetime.datetime(2026, 10, 23, 3, 34, 24, 21137), 654, -27.89)
  (datetime.datetime(2026, 10, 23, 0, 16, 39, 25787), 907, -19.09)
  (datetime.datetime(2026, 10, 16, 22, 53, 54, 23669), 1121, 0.36)
  (datetime.datetime(2026, 10, 16, 22, 51, 54, 22130), 1117, 0.36)


## Paso 5 — Validación de los archivos antes del despliegue

In [7]:
import ast

for archivo in ["scripts/ingesta.py", "scripts/publicar_gold.py",
                "dags/pipeline_grupo03.py"]:
    ruta = f"/home/jovyan/work/{archivo}"
    try:
        ast.parse(open(ruta, encoding="utf-8").read(), filename=ruta)
        estado = "OK"
    except SyntaxError as e:
        estado = f"ERROR linea {e.lineno}: {e.msg}"
    print(f"{archivo:35s} {estado}")

scripts/ingesta.py                  OK
scripts/publicar_gold.py            OK
dags/pipeline_grupo03.py            OK


## Paso 6 — Diagnóstico de la ejecución en Airflow

El DAG `pipeline_grupo03` encadena cuatro tareas:

```
ingerir_bronze -> dbt_run -> dbt_test -> publicar_gold
```

Al ejecutarlo, la primera tarea falla con:

```
ModuleNotFoundError: No module named 'deltalake'
```

**Verificación directa.** Se construyó un DAG de diagnóstico que inspecciona el
contenedor de Airflow. Resultado:

```
dbt NO INSTALADO
  duckdb: FALTA
  deltalake: FALTA
  boto3: OK | pandas: OK | requests: OK
  sqlalchemy: OK | psycopg2: OK | pyarrow: OK
```

**Diagnóstico.** La imagen de Airflow del servidor no incluye las dependencias
de transformación, aunque el `requirements.txt` del repositorio de la cátedra sí
las declara. Además, nuestro dominio usa Delta Lake, que no figura en ese
archivo.

**Qué sí funciona.** El DAG se parsea correctamente, carga el `.env`, resuelve
las rutas del workspace y encuentra los scripts: la falla ocurre al importar la
librería ausente, no antes. El pipeline completo se ejecuta en verde de forma
manual, como muestran las celdas anteriores.

**Corrección pendiente.** Reconstruir la imagen de Airflow con `dbt-core`,
`dbt-duckdb`, `duckdb` y `deltalake`. Reportado al docente.

**Correcciones ya aplicadas durante el diagnóstico**, en orden:

| Falla observada | Verificación directa | Corrección aplicada |
|---|---|---|
| `ModuleNotFoundError: entorno` | El contenedor no veía el módulo | Se copió `entorno.py` a `dags/` y se expuso la raíz en `PYTHONPATH` |
| Airflow marcaba `entorno.py` como DAG roto | Banner de Import Error en la interfaz | Se agregó `.airflowignore` |
| `./.env: Permission denied` | Log de la tarea `ingerir_bronze` | Permisos ajustados: Airflow corre con otro usuario |
| Faltaba `profiles.yml` en la copia de `dags/` | DAG de diagnóstico | Se unificó el proyecto dbt con un enlace |

## Cierre

| Componente | Estado |
|---|---|
| `scripts/ingesta.py` | ejecutable e idempotente, verificado |
| `scripts/publicar_gold.py` | publica en PostgreSQL, verificado |
| Proyecto dbt (staging + marts) | 13 nodos, todos en verde |
| `dags/pipeline_grupo03.py` | 4 tareas encadenadas, sintaxis validada |
| Ejecución completa en Airflow | bloqueada por dependencias de la imagen |

La documentación del pipeline está en `README_GRUPO.md`.